# House Price Prediction

## Notebook 3 — Feature Engineering

### Objectives

- Load cleaned dataset
- Separate features and target
- Encode categorical variables
- Split training and validation data
- Save processed datasets

Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

Paths

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED = ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

Load Clean Data

In [3]:
train = pd.read_csv(
    PROCESSED / "train_clean.csv"
)

train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,Gd,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,Gd,MnPrv,Shed,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Grvl,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,BrkFace,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,Gd,MnPrv,Shed,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,Grvl,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,Gd,MnPrv,Shed,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,Grvl,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,BrkFace,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,Gd,MnPrv,Shed,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,Grvl,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,Gd,MnPrv,Shed,0,12,2008,WD,Normal,250000


Shape

In [4]:
print(train.shape)

(1460, 81)


Separate Target

In [5]:
y = train["SalePrice"]

X = train.drop(
    columns=["SalePrice"]
)

Check Feature Types

In [6]:
X.dtypes.value_counts()

object     43
int64      34
float64     3
Name: count, dtype: int64

Find Categorical Columns

In [7]:
categorical_columns = X.select_dtypes(
    include="object"
).columns

print(f"Categorical columns: {len(categorical_columns)}")

categorical_columns

Categorical columns: 43


Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object')

One-Hot Encoding

In [8]:
X = pd.get_dummies(
    X,
    drop_first=True
)

Verify

In [9]:
X.dtypes.value_counts()

bool       208
int64       34
float64      3
Name: count, dtype: int64

Train/Validation Split

In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Shapes

In [11]:
print("X_train:", X_train.shape)

print("X_valid:", X_valid.shape)

print("y_train:", y_train.shape)

print("y_valid:", y_valid.shape)

X_train: (1168, 245)
X_valid: (292, 245)
y_train: (1168,)
y_valid: (292,)


Save CSVs

In [12]:
X_train.to_csv(
    PROCESSED / "X_train.csv",
    index=False
)

X_valid.to_csv(
    PROCESSED / "X_valid.csv",
    index=False
)

y_train.to_csv(
    PROCESSED / "y_train.csv",
    index=False
)

y_valid.to_csv(
    PROCESSED / "y_valid.csv",
    index=False
)

print("Datasets saved successfully.")

Datasets saved successfully.


Reload (Verification)

In [13]:
X_train = pd.read_csv(
    PROCESSED / "X_train.csv"
)

X_valid = pd.read_csv(
    PROCESSED / "X_valid.csv"
)

y_train = pd.read_csv(
    PROCESSED / "y_train.csv"
).squeeze()

y_valid = pd.read_csv(
    PROCESSED / "y_valid.csv"
).squeeze()

print(X_train.shape)
print(X_valid.shape)

(1168, 245)
(292, 245)


Final Check

In [14]:
print(X_train.head())

     Id  MSSubClass  LotFrontage  LotArea  OverallQual  OverallCond  \
0   255          20         70.0     8400            5            6   
1  1067          60         59.0     7837            6            7   
2   639          30         67.0     8777            5            7   
3   800          50         60.0     7200            5            7   
4   381          50         50.0     5000            5            6   

   YearBuilt  YearRemodAdd  MasVnrArea  BsmtFinSF1  BsmtFinSF2  BsmtUnfSF  \
0       1957          1957         0.0         922           0        392   
1       1993          1994         0.0           0           0        799   
2       1910          1950         0.0           0           0        796   
3       1937          1950       252.0         569           0        162   
4       1924          1950         0.0         218           0        808   

   TotalBsmtSF  1stFlrSF  2ndFlrSF  LowQualFinSF  GrLivArea  BsmtFullBath  \
0         1314      1314         